In [1]:

#STEP 1: Data Loading, Feature Integration, and Preprocessing
# Data.tsv is stored locally in the 
# same directory as of this python file
import pandas as pd
df = pd.read_csv('trainingSet.tsv',sep = '\t') 
display(df)

,bond,id,label_mutagenic
0,http://dl-learner.org/carcinogenesis#d187,3.0,1.0
1,http://dl-learner.org/carcinogenesis#d133,4.0,0.0
2,http://dl-learner.org/carcinogenesis#d23_2,6.0,0.0
3,http://dl-learner.org/carcinogenesis#d312,8.0,0.0
4,http://dl-learner.org/carcinogenesis#d81,10.0,0.0
...,...,...,...
267,http://dl-learner.org/carcinogenesis#d145,336.0,1.0
268,http://dl-learner.org/carcinogenesis#d94,337.0,0.0
269,http://dl-learner.org/carcinogenesis#d175,338.0,0.0
270,http://dl-learner.org/carcinogenesis#d34,339.0,0.0


In [4]:
# STEP 2:Install Required Dependencies for RDF Processing

!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/615.4 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 6.7 MB/s  0:00:00


In [ ]:
import os
import pandas as pd
from rdflib import Graph, Namespace, RDF, URIRef
from collections import Counter

INPUT_FILE = "mutag_stripped.nt"
OUTPUT_FILE = "mutag_compound_features.csv"

CARC = Namespace("http://dl-learner.org/carcinogenesis#")

def local_name(x):
    return str(x).split("#")[-1].split("/")[-1]

# Load RDF
g = Graph()
g.parse(INPUT_FILE, format="nt")

print("Triples loaded:", len(g), flush=True)

# Step 3: get real molecules/compounds only
compounds = sorted(set(g.subjects(RDF.type, CARC.Compound)), key=str)

print("Number of compounds:", len(compounds), flush=True)

rows = []

# Step 4: build one feature row per compound
for i, compound in enumerate(compounds):
    row = {
        "molecule": local_name(compound)
    }

    features = Counter()

    # Atom features
    atoms = list(g.objects(compound, CARC.hasAtom))
    row["num_atoms"] = len(atoms)

    for atom in atoms:
        for atom_type in g.objects(atom, RDF.type):
            features[f"atom_type__{local_name(atom_type)}"] += 1

        for charge in g.objects(atom, CARC.charge):
            features[f"charge__{local_name(charge)}"] += 1

    # Bond features
    bonds = list(g.objects(compound, CARC.hasBond))
    row["num_bonds"] = len(bonds)

    for bond in bonds:
        for bond_type in g.objects(bond, RDF.type):
            features[f"bond_type__{local_name(bond_type)}"] += 1

    # Structure features
    structures = list(g.objects(compound, CARC.hasStructure))
    row["num_structures"] = len(structures)

    for structure in structures:
        for structure_type in g.objects(structure, RDF.type):
            features[f"structure_type__{local_name(structure_type)}"] += 1

    # Compound-level test/label features
    for p, o in g.predicate_objects(compound):
        pred = local_name(p)

        if pred in ["hasAtom", "hasBond", "hasStructure", "type"]:
            continue

        obj = local_name(o)
        row[f"{pred}"] = obj

    # Add counted features
    row.update(features)

    rows.append(row)

    if i % 25 == 0:
        print("Processed compounds:", i, flush=True)

# Step 5: convert to DataFrame
df = pd.DataFrame(rows).fillna(0)

# Step 6: convert numeric feature columns to small integers
for col in df.columns:
    if col != "molecule":
        if col.startswith(("atom_type__", "bond_type__", "structure_type__", "charge__")):
            df[col] = df[col].astype("uint8")

print("Final shape:", df.shape, flush=True)
print(df.head(), flush=True)

# Step 7: save only small compound-level matrix
df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE, flush=True)
print("File size MB:", os.path.getsize(OUTPUT_FILE) / (1024 ** 2))

In [ ]:

# STEP 6:Merge features with train/test

import pandas as pd

TRAIN_FILE = "trainingSet.tsv"
TEST_FILE = "testSet.tsv"
FEATURE_FILE = "mutag_compound_features.csv"

train_df = pd.read_csv(TRAIN_FILE, sep="\t")
test_df = pd.read_csv(TEST_FILE, sep="\t")

features_df = pd.read_csv(FEATURE_FILE)

# Make molecule IDs consistent
features_df = features_df.rename(columns={"molecule": "bond"})

train_df["bond"] = train_df["bond"].astype(str)
test_df["bond"] = test_df["bond"].astype(str)
features_df["bond"] = features_df["bond"].astype(str)

# Merge train/test labels with molecule-level features
Xtrain = train_df.merge(features_df, on="bond", how="left")
Xtest = test_df.merge(features_df, on="bond", how="left")

# Replace missing feature values
Xtrain = Xtrain.fillna(0)
Xtest = Xtest.fillna(0)

# Separate labels
ytrain = Xtrain["label_mutagenic"]
ytest = Xtest["label_mutagenic"]

# Remove non-feature columns
drop_cols = ["id", "bond", "label_mutagenic"]

Xtrain_features = Xtrain.drop(columns=drop_cols, errors="ignore")
Xtest_features = Xtest.drop(columns=drop_cols, errors="ignore")

# One-hot encode non-numeric columns, if any
Xtrain_features = pd.get_dummies(Xtrain_features)
Xtest_features = pd.get_dummies(Xtest_features)

# Make train/test columns identical
Xtrain_features, Xtest_features = Xtrain_features.align(
    Xtest_features,
    join="left",
    axis=1,
    fill_value=0
)

print("Xtrain shape:", Xtrain.shape)
print("Xtest shape:", Xtest.shape)

print("Xtrain_features shape:", Xtrain_features.shape)
print("Xtest_features shape:", Xtest_features.shape)

print(Xtrain.head())
print(Xtest.head())

In [ ]:
# ============================================================
# STEP 7: Train Linear Regression Model
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# Initialize the Linear Regression model
lr_model = LinearRegression()

# Train the model
lr_model.fit(Xtrain_features, ytrain)

# Predict continuous values
lr_predictions = lr_model.predict(Xtest_features)

# Convert continuous predictions to binary classes
lr_predictions = np.where(lr_predictions >= 0.5, 1, 0)

# Evaluate the model
lr_accuracy = accuracy_score(ytest, lr_predictions)

print("Linear Regression Accuracy:", lr_accuracy)

print("\nClassification Report:")
print(classification_report(ytest, lr_predictions, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, lr_predictions))

In [ ]:
# STEP 9:Train SVM model

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Initialize the SVM classifier
svm_model = SVC(
    kernel="linear",
    random_state=42
)

# Train the model
svm_model.fit(Xtrain_features, ytrain)

# Make predictions
svm_predictions = svm_model.predict(Xtest_features)

# Evaluate the model
accuracy = accuracy_score(ytest, svm_predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(ytest, svm_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, svm_predictions))

In [ ]:
# STEP 10: Train Decision Tree model



from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Initialize Decision Tree classifier
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

# Train the model
dt_model.fit(Xtrain_features, ytrain)

# Make predictions
dt_predictions = dt_model.predict(Xtest_features)

# Evaluate the model
dt_accuracy = accuracy_score(ytest, dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)

print("\nClassification Report:")
print(classification_report(ytest, dt_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, dt_predictions))

In [ ]:
# STEP 11:Compare Model Performance

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

# Evaluate all models
results = [
    evaluate_model("Linear Regression", ytest, lr_predictions),
    evaluate_model("SVM", ytest, svm_predictions),
    evaluate_model("Decision Tree", ytest, dt_predictions)
]

# Create comparison table
results_df = pd.DataFrame(results)

print("\n========== Model Performance Comparison ==========\n")
print(results_df.to_string(index=False))

# Identify best model based on F1-score
max_f1 = results_df["F1 Score"].max()
best_models = results_df[results_df["F1 Score"] == max_f1]

print("\nBest Performing Model:")

if len(best_models) > 1:
    print("Multiple models performed equally based on F1 Score.")
    print(best_models.to_string(index=False))
else:
    best_model = best_models.iloc[0]
    print(f"Model     : {best_model['Model']}")
    print(f"Accuracy  : {best_model['Accuracy']:.4f}")
    print(f"Precision : {best_model['Precision']:.4f}")
    print(f"Recall    : {best_model['Recall']:.4f}")
    print(f"F1 Score  : {best_model['F1 Score']:.4f}")

In [ ]:
# STEP 12: Classification reports



from sklearn.metrics import classification_report
import numpy as np

# Get class names from the test labels
class_names = [str(c) for c in sorted(np.unique(ytest))]

# Linear Regression Classification Report
print("\n========== Linear Regression Classification Report ==========\n")
print(
    classification_report(
        ytest,
        lr_predictions,
        target_names=class_names,
        zero_division=0
    )
)

# SVM Classification Report
print("\n========== SVM Classification Report ==========\n")
print(
    classification_report(
        ytest,
        svm_predictions,
        target_names=class_names,
        zero_division=0
    )
)

# Decision Tree Classification Report
print("\n========== Decision Tree Classification Report ==========\n")
print(
    classification_report(
        ytest,
        dt_predictions,
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
# STEP 13: Confusion matrix 


from sklearn.metrics import confusion_matrix

print("Linear Regression Confusion Matrix:")
print(confusion_matrix(ytest, lr_predictions))

print("SVM Confusion Matrix:")
print(confusion_matrix(ytest, svm_predictions))

print("\nDecision Tree Confusion Matrix:")
print(confusion_matrix(ytest, dt_predictions))

In [ ]:
# STEP 14: Save outputs


import os

# Save feature matrix
feature_file = "mutag_compound_features.csv"
features_df.to_csv(feature_file, index=False)

# Save model performance comparison
if 'results_df' in locals():
    performance_file = "model_performance_comparison.csv"
    results_df.to_csv(performance_file, index=False)

    print("\nFiles saved successfully:")
    print(f"1. {feature_file}")
    print(f"2. {performance_file}")

    print(f"\nFeature Matrix Size: {os.path.getsize(feature_file)/(1024**2):.2f} MB")
    print(f"Performance File Size: {os.path.getsize(performance_file)/(1024**2):.2f} MB")

else:
    print("\nFeature matrix saved successfully.")
    print(f"File: {feature_file}")
    print("\nresults_df not found. Please run the model evaluation step before saving the performance comparison.")